# MintPy Euler Pole object Unit test

#### Yuan-Kai Liu Oct 18, 2022

## Section 1: Regular test (this part is also placed at `MintPy/tests/objects/euler.py`)


In [2]:
PMM="""
ITRF2014 No-net-rotation Plate Motion Model (Altamimi et al., 2017)
Tag     name           num_site  omega_x     omega_y     omega_z     omega     wrms_e    wrms_n    
ANTA    Antartica        7      -0.248      -0.324       0.675       0.219       0.200       0.160
ARAB    Arabia           5       1.154      -0.136       1.444       0.515       0.360       0.430
AUST    Australia       36       1.510       1.182       1.215       0.631       0.240       0.200
EURA    Eurasia         97      -0.085      -0.531       0.770       0.261       0.230       0.190
INDI    India            3       1.154      -0.005       1.454       0.516       0.210       0.210
NAZC    Nazca            2      -0.333      -1.544       1.623       0.629       0.130       0.190
NOAM    N. America      72       0.024      -0.694      -0.063       0.194       0.230       0.280
NUBI    Nubia           24       0.099      -0.614       0.733       0.267       0.280       0.360
PCFC    Pacific         18      -0.409       1.047      -2.169       0.679       0.360       0.310
SOAM    S. America      30      -0.270      -0.301      -0.140       0.119       0.340       0.350
SOMA    Somalia          3      -0.121      -0.794       0.884       0.332       0.320       0.300
"""

In [8]:
#!/usr/bin/env python3
# Author: Yuan-Kai Liu, Zhang Yunjun, Oct 2022
"""Test mintpy.objects.euler module for the Euler pole and velocity computation."""


import collections
import math

import numpy as np
from mintpy.objects.euler_pole import sph2cart
from mintpy.utils.utils0 import calc_azimuth_from_east_north_obs

from mintpy.objects.euler_pole import MASY2DMY, EulerPole
from mintpy.plate_motion import ITRF2014_PMM

# validation against the UNAVCO Plate Motion Calculator
# https://www.unavco.org/software/geodetic-utilities/plate-motion-calculator/plate-motion-calculator.html
# accessed on Oct 29, 2022.
# config: height=0, model=ITRF2014, reference=NNR no-net-rotation
# note: azimuth here is measured from the north with positive for clock-wise direction
# unit:                              deg deg mm/yr   deg   mm/yr mm/yr
Tag = collections.namedtuple('Tag', 'lat lon speed azimuth vel_n vel_e')
POINT_PM = {
    'Australia' : Tag(-24,  132,  67.56,  28.94,  59.12,  32.69),
    'Eurasia' : Tag( 27,   62,  28.85,  79.21,   5.40,  28.34),
    'Arabia' : Tag( 18,   48,  46.51,  50.92,  29.32,  36.11),
}
PLATE_NAMES = POINT_PM.keys()


def test_euler_pole_initiation():
    print('Test 1: EulerPole object initiation and vector2pole conversion.')

    for plate_name in PLATE_NAMES:
        print(f'Plate name: ITRF2014-PMM {plate_name}')

        # get PMM info from Table 1 in Altamimi et al. (2017) as reference
        plate_pmm = ITRF2014_PMM[plate_name]

        # build EulerPole obj
        pole_obj = EulerPole(wx=plate_pmm.omega_x, wy=plate_pmm.omega_y, wz=plate_pmm.omega_z)

        # compare rotation rate: ITRF2014_PMM vs. Euler vector to pole conversion
        print(f'Reference  rotation rate from Altamimi et al. (2017): {plate_pmm.omega:.4f} deg/Ma')
        print(f'Calculated rotation rate from pole2vector conversion: {plate_pmm.omega:.4f} deg/Ma')
        assert math.isclose(plate_pmm.omega, pole_obj.rotRate*MASY2DMY, abs_tol=5e-4)
        print('Pass.')


def test_plate_motion_calc():
    print('Test 2: Plate motion calculation and validation against UNAVCO website.')

    for plate_name in PLATE_NAMES:
        # get UNAVCO result as reference
        point_pm = POINT_PM[plate_name]
        print(f'Plate = ITRF2014-PMM {plate_name}, point lat/lon = {point_pm.lat}/{point_pm.lon}')

        # calculate using EulerPole in m/yr
        plate_pmm = ITRF2014_PMM[plate_name]
        pole_obj = EulerPole(wx=plate_pmm.omega_x, wy=plate_pmm.omega_y, wz=plate_pmm.omega_z)
        ve, vn = pole_obj.get_velocity_enu(point_pm.lat, point_pm.lon, print_msg=False)[:2]

        print(f'Reference   (UNAVCO): vel_e={point_pm.vel_e:.2f}, vel_n={point_pm.vel_n:.2f} mm/yr')
        print(f'Calculation (MintPy): vel_e={ve*1e3:.2f}, vel_n={vn*1e3:.2f} mm/yr')
        assert math.isclose(point_pm.vel_e, ve*1e3, abs_tol=0.05)
        assert math.isclose(point_pm.vel_n, vn*1e3, abs_tol=0.05)
        print('Pass.')


if __name__ == '__main__':

    print('-'*50)
    #print(f'Testing {__file__}')

    test_euler_pole_initiation()

    test_plate_motion_calc()


--------------------------------------------------
Test 1: EulerPole object initiation and vector2pole conversion.
Plate name: ITRF2014-PMM Australia
Reference  rotation rate from Altamimi et al. (2017): 0.6310 deg/Ma
Calculated rotation rate from pole2vector conversion: 0.6310 deg/Ma
Pass.
Plate name: ITRF2014-PMM Eurasia
Reference  rotation rate from Altamimi et al. (2017): 0.2610 deg/Ma
Calculated rotation rate from pole2vector conversion: 0.2610 deg/Ma
Pass.
Plate name: ITRF2014-PMM Arabia
Reference  rotation rate from Altamimi et al. (2017): 0.5150 deg/Ma
Calculated rotation rate from pole2vector conversion: 0.5150 deg/Ma
Pass.
Test 2: Plate motion calculation and validation against UNAVCO website.
Plate = ITRF2014-PMM Australia, point lat/lon = -24/132
Reference   (UNAVCO): vel_e=32.69, vel_n=59.12 mm/yr
Calculation (MintPy): vel_e=32.69, vel_n=59.12 mm/yr
Pass.
Plate = ITRF2014-PMM Eurasia, point lat/lon = 27/62
Reference   (UNAVCO): vel_e=28.34, vel_n=5.40 mm/yr
Calculation (Mi

## Section 1.1 test some relative motion

In [52]:
name  = 'Sinai'
dname = 'Sinai'

# Castro-Perdomo et al., GJI, 2022
# the Euler pole of the Sinai subplate with respect to ITRF2014 using five reliable stations 
# located on the Sinai subplate at distances greater than 40 km from the DST (CSAR, ALON, YRCM, BSHM and RAMO)
# The SINAI-ITRF2014 Euler pole derived here is 54.7±0.7° N, 347.8±4.0° E, ω = 0.417± 0.021° Ma−1
om_sph = sph2cart(54.7, 347.8, r=0.417/MASY2DMY)
print('New plate:', name, om_sph, 'mas per year')


# Augment ITRF2014 PMM
TagPMM = collections.namedtuple('Tag', 'name num_site omega_x omega_y omega_z omega wrms_e wrms_n')
ITRF2014_PMM[dname] = TagPMM(name, None, om_sph[0], om_sph[1], om_sph[2], np.linalg.norm(om_sph), None, None)
#ITRF2014_PMM

# build EulerPole obj
arab_obj = EulerPole(wx=ITRF2014_PMM['Arabia'].omega_x, wy=ITRF2014_PMM['Arabia'].omega_y, wz=ITRF2014_PMM['Arabia'].omega_z)
eura_obj = EulerPole(wx=ITRF2014_PMM['Eurasia'].omega_x,wy=ITRF2014_PMM['Eurasia'].omega_y,wz=ITRF2014_PMM['Eurasia'].omega_z)
nubi_obj = EulerPole(wx=ITRF2014_PMM['Nubia'].omega_x,  wy=ITRF2014_PMM['Nubia'].omega_y,  wz=ITRF2014_PMM['Nubia'].omega_z)
sini_obj = EulerPole(wx=ITRF2014_PMM['Sinai'].omega_x,  wy=ITRF2014_PMM['Sinai'].omega_y,  wz=ITRF2014_PMM['Sinai'].omega_z)

# relative motion
rpm_NA = nubi_obj - arab_obj
rpm_SA = sini_obj - arab_obj
print('Relative Euler pole: Nubia-Arabia\n', rpm_NA)
print('Relative Euler pole: Sinai-Arabia\n', rpm_SA)

# Sinai-Arabia pole (Viltres et. al., 2021 Table 1)
rpm_SA2 = EulerPole(wx=-0.1055, wy=-0.0282, wz=-0.0803, unit='deg/Ma')

New plate: Sinai (0.8478886087323123, -0.18332000605338197, 1.2251857502283363) mas per year
Relative Euler pole: Nubia-Arabia
 EulerPole(name=None, poleLat=-31.544303466638066, poleLon=-155.62564348844015, rotRate=1.3590548186147606, wx=-1.055, wy=-0.478, wz=-0.711, unit=mas/yr)
Relative Euler pole: Sinai-Arabia
 EulerPole(name=None, poleLat=-35.23846747435657, poleLon=-171.21252919432177, rotRate=0.3792400858821064, wx=-0.3061113912676876, wy=-0.04732000605338196, wz=-0.21881424977166364, unit=mas/yr)


In [54]:
rpm_SA2

EulerPole(name=None, poleLat=-36.32785462609328, poleLon=-165.03478598775175, rotRate=0.48797704536176695, wx=-0.37979999999999997, wy=-0.10152, wz=-0.28907999999999995, unit=mas/yr)

## Section 2: Test on a grid for computation time

In [20]:
def get_lalo(lat0, lat1, lon0, lon1, N):
    """
    Parameters: lat0 - starting lat
                lat1 -   ending lat
                lon0 - starting lon
                lon1 -   ending lon
                N    - number of grids in both directions
    Returns:    lats - gridded lat
                lons - gridded lon
    """
    lats = np.linspace(lat0, lat1, N)
    lons = np.linspace(lon0,lon1, N)
    lats, lons = np.meshgrid(lats, lons)
    return lats, lons

lat0, lat1 = 30, 40
lon0, lon1 = 90, 110

In [23]:
%%time

lats, lons = get_lalo(lat0, lat1, lon0, lon1, 10)

v = 1e3 * np.array(eura_obj.get_velocity_enu(lats, lons))

assume a spheroidal Earth as defined in WGS84
CPU times: user 6.33 ms, sys: 1.09 ms, total: 7.41 ms
Wall time: 5.93 ms


In [24]:
%%time

lats, lons = get_lalo(lat0, lat1, lon0, lon1, 50)

v = 1e3 * np.array(eura_obj.get_velocity_enu(lats, lons))

assume a spheroidal Earth as defined in WGS84
CPU times: user 6.54 ms, sys: 2.14 ms, total: 8.68 ms
Wall time: 6.95 ms


In [25]:
%%time

lats, lons = get_lalo(lat0, lat1, lon0, lon1, 100)

v = 1e3 * np.array(eura_obj.get_velocity_enu(lats, lons))

assume a spheroidal Earth as defined in WGS84
CPU times: user 13.4 ms, sys: 2.09 ms, total: 15.5 ms
Wall time: 13.4 ms


In [26]:
%%time

lats, lons = get_lalo(lat0, lat1, lon0, lon1, 1000)

v = 1e3 * np.array(eura_obj.get_velocity_enu(lats, lons))

assume a spheroidal Earth as defined in WGS84
CPU times: user 662 ms, sys: 60.8 ms, total: 723 ms
Wall time: 722 ms


In [11]:
%%time

lats, lons = get_lalo(lat0, lat1, lon0, lon1, 10000)

v = 1e3 * np.array(eura_obj.get_velocity_enu(lats, lons))

Assume WGS84 ellipse from pyproj
number of points to compute: 100000000
CPU times: user 44.4 s, sys: 9.39 s, total: 53.8 s
Wall time: 53.9 s
